# 🌊 AISEHack Phase 2 — Complete EDA Notebook
**Goal:** Understand the data deeply before touching any model.

What this notebook covers:
1. Data structure & file loading
2. Image metadata (shape, dtype, CRS, nodata)
3. Channel statistics (min/max/mean/std per band)
4. Class distribution analysis (0=NoFlood, 1=Flood, 2=WaterBody)
5. Visual inspection of patches + labels
6. Spectral indices (NDWI, SAR ratio, NDVI)
7. NaN / nodata / corruption check
8. Patch-level class imbalance
9. Channel correlation matrix
10. Summary report — everything you need for modelling decisions

In [ ]:
# ── Install if needed ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'rasterio', '-q'])

In [ ]:
# ── Core Imports ───────────────────────────────────────────────────────────────
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

import rasterio
from rasterio.errors import NotGeoreferencedWarning
warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

print('✅ All imports successful')
print(f'   rasterio version: {rasterio.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE      = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
IMG_DIR   = os.path.join(BASE, 'image')
LBL_DIR   = os.path.join(BASE, 'label')
PRED_DIR  = os.path.join(BASE, 'prediction', 'image')
SPLIT_DIR = os.path.join(BASE, 'split')

for p in [IMG_DIR, LBL_DIR, PRED_DIR, SPLIT_DIR]:
    exists = os.path.exists(p)
    print(f'  {"✅" if exists else "❌"}  {p}')

In [ ]:
# ── Load Split Files ───────────────────────────────────────────────────────────
def load_split(fname):
    path = os.path.join(SPLIT_DIR, fname)
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_ids = load_split('train.txt')
val_ids   = load_split('val.txt')
test_ids  = load_split('test.txt')

# Also check pred.txt if exists
try:
    pred_ids = load_split('pred.txt')
except:
    pred_ids = []

print(f'Train patches : {len(train_ids)}')
print(f'Val   patches : {len(val_ids)}')
print(f'Test  patches : {len(test_ids)}')
print(f'Pred  patches : {len(pred_ids)}')
print(f'\nTotal with labels : {len(train_ids) + len(val_ids)}')
print(f'\nSample train IDs  : {train_ids[:3]}')
print(f'Sample test IDs   : {test_ids[:3]}')

In [ ]:
# ── File Discovery ─────────────────────────────────────────────────────────────
all_images = sorted(glob.glob(os.path.join(IMG_DIR, '*.tif')))
all_labels = sorted(glob.glob(os.path.join(LBL_DIR, '*.tif')))
all_preds  = sorted(glob.glob(os.path.join(PRED_DIR, '*.tif')))

print(f'Image TIFs found : {len(all_images)}')
print(f'Label TIFs found : {len(all_labels)}')
print(f'Pred  TIFs found : {len(all_preds)}')
print(f'\nSample image filename : {os.path.basename(all_images[0]) if all_images else "NONE"}')
print(f'Sample label filename : {os.path.basename(all_labels[0]) if all_labels else "NONE"}')

In [ ]:
# ── Helper: build ID-to-path mapping ──────────────────────────────────────────
# IDs in txt files may or may not have suffixes — let's be flexible
def build_path_map(file_list):
    d = {}
    for f in file_list:
        base = os.path.basename(f)
        # strip common suffixes
        key = base.replace('_image.tif','').replace('_label.tif','').replace('.tif','')
        d[key] = f
    return d

img_map  = build_path_map(all_images)
lbl_map  = build_path_map(all_labels)
pred_map = build_path_map(all_preds)

print(f'Image map keys (first 3): {list(img_map.keys())[:3]}')
print(f'Label map keys (first 3): {list(lbl_map.keys())[:3]}')
print(f'Pred  map keys (first 3): {list(pred_map.keys())[:3]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 1: IMAGE METADATA
# ═══════════════════════════════════════════════════════════════════

# Open first image and inspect everything
sample_img_path = all_images[0]

with rasterio.open(sample_img_path) as src:
    print('═' * 55)
    print('  IMAGE METADATA (from first TIF)')
    print('═' * 55)
    print(f'  File         : {os.path.basename(sample_img_path)}')
    print(f'  Width x Height : {src.width} x {src.height}')
    print(f'  Band count   : {src.count}')
    print(f'  Data types   : {src.dtypes}')
    print(f'  CRS          : {src.crs}')
    print(f'  Transform    : {src.transform}')
    print(f'  Bounds       : {src.bounds}')
    print(f'  NoData vals  : {src.nodatavals}')
    print(f'  Resolution   : {src.res}')
    data = src.read()  # shape: (bands, H, W)

print(f'\n  Numpy array shape: {data.shape}  → (bands, height, width)')
print(f'  dtype            : {data.dtype}')
print(f'  Overall min      : {data.min()}')
print(f'  Overall max      : {data.max()}')

In [ ]:
# Check label file metadata too
sample_lbl_path = all_labels[0]

with rasterio.open(sample_lbl_path) as src:
    print('═' * 55)
    print('  LABEL METADATA (from first label TIF)')
    print('═' * 55)
    print(f'  File         : {os.path.basename(sample_lbl_path)}')
    print(f'  Width x Height : {src.width} x {src.height}')
    print(f'  Band count   : {src.count}')
    print(f'  Data types   : {src.dtypes}')
    print(f'  NoData vals  : {src.nodatavals}')
    lbl = src.read(1)  # single band mask

print(f'\n  Label array shape  : {lbl.shape}')
print(f'  Unique label values: {np.unique(lbl)}')
print(f'  (Expected: 0, 1, 2 for NoFlood, Flood, WaterBody)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 2: CHANNEL STATISTICS ACROSS ALL LABELLED PATCHES
# ═══════════════════════════════════════════════════════════════════

# Channel naming — based on dataset description
# The TIF may have 6 or 7 bands depending on whether label is channel 0
# We'll detect this automatically

CHANNEL_NAMES_7 = ['FloodLabel', 'SAR_HH', 'SAR_HV', 'Green', 'Red', 'NIR', 'SWIR']
CHANNEL_NAMES_6 = ['SAR_HH', 'SAR_HV', 'Green', 'Red', 'NIR', 'SWIR']

# Detect number of bands
with rasterio.open(all_images[0]) as src:
    n_bands = src.count

if n_bands == 7:
    CHANNEL_NAMES = CHANNEL_NAMES_7
    IMG_BAND_START = 1  # bands to use for input (skip channel 0 = label)
elif n_bands == 6:
    CHANNEL_NAMES = CHANNEL_NAMES_6
    IMG_BAND_START = 0
else:
    CHANNEL_NAMES = [f'Band_{i}' for i in range(n_bands)]
    IMG_BAND_START = 0

print(f'Detected {n_bands} bands in image TIF')
print(f'Channel names assigned: {CHANNEL_NAMES}')

# ── Compute per-channel stats across all labelled patches ──────────
labelled_ids = train_ids + val_ids

stats = {name: {'min': [], 'max': [], 'mean': [], 'std': [], 'nan_count': []}
         for name in CHANNEL_NAMES}

missing = []
for pid in labelled_ids:
    # Find matching image file
    img_path = None
    for k, v in img_map.items():
        if pid in k or k in pid:
            img_path = v
            break
    if img_path is None:
        missing.append(pid)
        continue
    with rasterio.open(img_path) as src:
        arr = src.read().astype(np.float32)  # (bands, H, W)
    for i, name in enumerate(CHANNEL_NAMES):
        band = arr[i]
        stats[name]['min'].append(np.nanmin(band))
        stats[name]['max'].append(np.nanmax(band))
        stats[name]['mean'].append(np.nanmean(band))
        stats[name]['std'].append(np.nanstd(band))
        stats[name]['nan_count'].append(int(np.isnan(band).sum()))

print(f'\nProcessed {len(labelled_ids) - len(missing)} patches (missing: {len(missing)})')

# Build summary DataFrame
rows = []
for name in CHANNEL_NAMES:
    rows.append({
        'Channel'  : name,
        'GlobalMin': np.min(stats[name]['min']),
        'GlobalMax': np.max(stats[name]['max']),
        'MeanOfMeans': np.mean(stats[name]['mean']),
        'MeanOfStds' : np.mean(stats[name]['std']),
        'TotalNaNs'  : np.sum(stats[name]['nan_count']),
    })

stats_df = pd.DataFrame(rows)
print('\n', stats_df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 3: CLASS DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════

CLASS_NAMES  = {0: 'NoFlood', 1: 'Flood', 2: 'WaterBody'}
CLASS_COLORS = {0: '#a8d5a2', 1: '#e63946', 2: '#457b9d'}

global_counts = {0: 0, 1: 0, 2: 0}
patch_class_pcts = []  # per-patch class percentages

for pid in labelled_ids:
    lbl_path = None
    for k, v in lbl_map.items():
        if pid in k or k in pid:
            lbl_path = v
            break
    if lbl_path is None:
        continue
    with rasterio.open(lbl_path) as src:
        lbl = src.read(1)
    total = lbl.size
    row = {'patch_id': pid}
    for c in [0, 1, 2]:
        cnt = int((lbl == c).sum())
        global_counts[c] += cnt
        row[CLASS_NAMES[c]] = round(cnt / total * 100, 2)
    patch_class_pcts.append(row)

patch_df = pd.DataFrame(patch_class_pcts)

total_pixels = sum(global_counts.values())
print('═' * 55)
print('  GLOBAL CLASS DISTRIBUTION')
print('═' * 55)
for c, name in CLASS_NAMES.items():
    cnt = global_counts[c]
    pct = cnt / total_pixels * 100
    print(f'  Class {c} ({name:10s}): {cnt:>12,} pixels  ({pct:.2f}%)')
print(f'  Total pixels        : {total_pixels:>12,}')

print('\n  PER-PATCH CLASS % — SUMMARY STATS')
print(patch_df[['NoFlood','Flood','WaterBody']].describe().round(2).to_string())

In [ ]:
# ── Class Distribution Plots ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Class Distribution Analysis', fontsize=14, fontweight='bold')

# 1. Global pie chart
ax = axes[0]
labels = [f"{CLASS_NAMES[c]}\n({global_counts[c]/total_pixels*100:.1f}%)" for c in [0,1,2]]
sizes  = [global_counts[c] for c in [0,1,2]]
colors = [CLASS_COLORS[c] for c in [0,1,2]]
ax.pie(sizes, labels=labels, colors=colors, startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Global Pixel Distribution')

# 2. Per-patch flood % histogram
ax = axes[1]
ax.hist(patch_df['Flood'], bins=20, color=CLASS_COLORS[1], edgecolor='white', alpha=0.85)
ax.axvline(patch_df['Flood'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f"Mean={patch_df['Flood'].mean():.1f}%")
ax.set_xlabel('Flood pixel % in patch')
ax.set_ylabel('Number of patches')
ax.set_title('Per-Patch Flood Percentage')
ax.legend()

# 3. Per-patch water body % histogram
ax = axes[2]
ax.hist(patch_df['WaterBody'], bins=20, color=CLASS_COLORS[2], edgecolor='white', alpha=0.85)
ax.axvline(patch_df['WaterBody'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f"Mean={patch_df['WaterBody'].mean():.1f}%")
ax.set_xlabel('WaterBody pixel % in patch')
ax.set_ylabel('Number of patches')
ax.set_title('Per-Patch WaterBody Percentage')
ax.legend()

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

In [ ]:
# ─ Which patches have the most flood? (top 10 for model focus)
top_flood = patch_df.nlargest(10, 'Flood')[['patch_id','NoFlood','Flood','WaterBody']]
print('TOP 10 PATCHES BY FLOOD %:')
print(top_flood.to_string(index=False))

print('\nPATCHES WITH ZERO FLOOD (pure background):')
zero_flood = patch_df[patch_df['Flood'] == 0.0]
print(f'  {len(zero_flood)} patches have 0% flood pixels')
print(f'  {len(patch_df[patch_df["WaterBody"] == 0.0])} patches have 0% water body pixels')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 4: VISUAL INSPECTION — 3 patches side by side
# ═══════════════════════════════════════════════════════════════════

def load_image_label(pid, img_map, lbl_map):
    img_path = lbl_path = None
    for k, v in img_map.items():
        if pid in k or k in pid:
            img_path = v; break
    for k, v in lbl_map.items():
        if pid in k or k in pid:
            lbl_path = v; break
    if not img_path or not lbl_path:
        return None, None
    with rasterio.open(img_path) as src:
        img = src.read().astype(np.float32)
    with rasterio.open(lbl_path) as src:
        lbl = src.read(1)
    return img, lbl

def normalize_band(band):
    """Clip to 2nd-98th percentile and scale 0-1."""
    p2, p98 = np.nanpercentile(band, 2), np.nanpercentile(band, 98)
    band = np.clip(band, p2, p98)
    if p98 > p2:
        band = (band - p2) / (p98 - p2)
    return band

def make_label_rgb(lbl):
    """Convert 0/1/2 label to RGB for display."""
    h, w = lbl.shape
    rgb = np.zeros((h, w, 3), dtype=np.float32)
    # 0 = NoFlood → light green
    rgb[lbl == 0] = [0.659, 0.835, 0.635]
    # 1 = Flood → red
    rgb[lbl == 1] = [0.902, 0.224, 0.275]
    # 2 = WaterBody → blue
    rgb[lbl == 2] = [0.271, 0.482, 0.616]
    return rgb

# Pick 3 interesting patches: one flood-heavy, one waterbody-heavy, one mixed
if len(patch_df) > 0:
    pid_flood  = patch_df.nlargest(1, 'Flood')['patch_id'].values[0]
    pid_water  = patch_df.nlargest(1, 'WaterBody')['patch_id'].values[0]
    pid_mixed  = patch_df[(patch_df['Flood'] > 1) & 
                          (patch_df['WaterBody'] > 1)].iloc[0]['patch_id'] if len(
                  patch_df[(patch_df['Flood'] > 1) & (patch_df['WaterBody'] > 1)]) > 0 \
                  else labelled_ids[0]
    selected_pids = [pid_flood, pid_water, pid_mixed]
    titles        = ['Most Flood', 'Most WaterBody', 'Mixed']
else:
    selected_pids = labelled_ids[:3]
    titles = ['Patch 1', 'Patch 2', 'Patch 3']

# Band indices for SAR and optical
# If 7-band: [0]=label,[1]=HH,[2]=HV,[3]=G,[4]=R,[5]=NIR,[6]=SWIR
# If 6-band: [0]=HH,[1]=HV,[2]=G,[3]=R,[4]=NIR,[5]=SWIR
if n_bands == 7:
    idx_HH, idx_HV = 1, 2
    idx_G, idx_R, idx_NIR, idx_SWIR = 3, 4, 5, 6
else:
    idx_HH, idx_HV = 0, 1
    idx_G, idx_R, idx_NIR, idx_SWIR = 2, 3, 4, 5

fig, axes = plt.subplots(len(selected_pids), 5,
                         figsize=(22, 5 * len(selected_pids)))
if len(selected_pids) == 1:
    axes = [axes]

legend_patches = [
    mpatches.Patch(color='#a8d5a2', label='0 – NoFlood'),
    mpatches.Patch(color='#e63946', label='1 – Flood'),
    mpatches.Patch(color='#457b9d', label='2 – WaterBody'),
]

for row_i, (pid, title) in enumerate(zip(selected_pids, titles)):
    img, lbl = load_image_label(pid, img_map, lbl_map)
    if img is None:
        print(f'⚠️  Could not load {pid}')
        continue

    axes[row_i][0].set_ylabel(f'{title}\n{pid}', fontsize=8)

    # Col 0: RGB composite (Green, Red, NIR → pseudo-color)
    try:
        rgb = np.stack([
            normalize_band(img[idx_NIR]),  # R channel → NIR for false color
            normalize_band(img[idx_R]),
            normalize_band(img[idx_G]),
        ], axis=-1)
        axes[row_i][0].imshow(rgb)
        axes[row_i][0].set_title('False Color (NIR-R-G)')
    except Exception as e:
        axes[row_i][0].text(0.5, 0.5, str(e), ha='center', va='center')

    # Col 1: SAR HH
    axes[row_i][1].imshow(normalize_band(img[idx_HH]), cmap='gray')
    axes[row_i][1].set_title('SAR HH')

    # Col 2: SAR HV
    axes[row_i][2].imshow(normalize_band(img[idx_HV]), cmap='gray')
    axes[row_i][2].set_title('SAR HV')

    # Col 3: NDWI = (Green - NIR) / (Green + NIR)
    G   = img[idx_G].astype(np.float32)
    NIR = img[idx_NIR].astype(np.float32)
    ndwi = np.where((G + NIR) != 0, (G - NIR) / (G + NIR + 1e-8), 0)
    im3 = axes[row_i][3].imshow(ndwi, cmap='RdYlBu', vmin=-1, vmax=1)
    axes[row_i][3].set_title('NDWI (water index)')
    plt.colorbar(im3, ax=axes[row_i][3], fraction=0.046, pad=0.04)

    # Col 4: Label
    axes[row_i][4].imshow(make_label_rgb(lbl))
    axes[row_i][4].set_title('Ground Truth Label')
    axes[row_i][4].legend(handles=legend_patches, loc='lower right', fontsize=7)

    for ax in axes[row_i]:
        ax.axis('off')

plt.suptitle('Visual Inspection: SAR + Optical + NDWI + Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('visual_inspection.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: visual_inspection.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 5: ALL 6 CHANNELS on one patch
# ═══════════════════════════════════════════════════════════════════

img, lbl = load_image_label(selected_pids[0], img_map, lbl_map)

n_plot_bands = min(n_bands, len(CHANNEL_NAMES))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

cmaps = ['gray', 'gray', 'YlGn', 'Reds', 'RdYlGn', 'Blues', 'Purples', 'Oranges']

for i in range(n_plot_bands):
    band = img[i]
    ax = axes[i]
    im = ax.imshow(normalize_band(band), cmap=cmaps[i % len(cmaps)])
    ax.set_title(f'Band {i}: {CHANNEL_NAMES[i]}', fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Last subplot = label
axes[n_plot_bands].imshow(make_label_rgb(lbl))
axes[n_plot_bands].set_title('Ground Truth (3-class)', fontsize=11)
axes[n_plot_bands].legend(handles=legend_patches, loc='lower right', fontsize=8)
axes[n_plot_bands].axis('off')

for j in range(n_plot_bands + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'All Channels — {selected_pids[0]}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('all_channels.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: all_channels.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 6: SPECTRAL INDICES ANALYSIS
# ═══════════════════════════════════════════════════════════════════
# Key question: Do these indices separate classes well?

index_stats = {cls: {'ndwi': [], 'sar_ratio': [], 'ndvi': [], 'swir_nir': []}
               for cls in [0, 1, 2]}

for pid in labelled_ids[:30]:  # use first 30 patches for speed
    img, lbl = load_image_label(pid, img_map, lbl_map)
    if img is None:
        continue

    G    = img[idx_G].astype(np.float64)
    R    = img[idx_R].astype(np.float64)
    NIR  = img[idx_NIR].astype(np.float64)
    SWIR = img[idx_SWIR].astype(np.float64)
    HH   = img[idx_HH].astype(np.float64)
    HV   = img[idx_HV].astype(np.float64)

    ndwi      = (G - NIR)  / (G + NIR + 1e-8)
    ndvi      = (NIR - R)  / (NIR + R + 1e-8)
    sar_ratio = HH / (HV + 1e-8)
    swir_nir  = SWIR / (NIR + 1e-8)

    for cls in [0, 1, 2]:
        mask = (lbl == cls)
        if mask.sum() == 0:
            continue
        # Sample max 2000 pixels per class per patch for speed
        indices = np.argwhere(mask)
        if len(indices) > 2000:
            indices = indices[np.random.choice(len(indices), 2000, replace=False)]
        rows, cols = indices[:, 0], indices[:, 1]

        index_stats[cls]['ndwi'].extend(ndwi[rows, cols].tolist())
        index_stats[cls]['sar_ratio'].extend(sar_ratio[rows, cols].tolist())
        index_stats[cls]['ndvi'].extend(ndvi[rows, cols].tolist())
        index_stats[cls]['swir_nir'].extend(swir_nir[rows, cols].tolist())

print('Spectral index samples collected per class:')
for cls in [0,1,2]:
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {len(index_stats[cls]["ndwi"]):,} samples')

In [ ]:
# ── Box plots of indices by class ──────────────────────────────────
index_labels = ['NDWI', 'SAR_Ratio (HH/HV)', 'NDVI', 'SWIR/NIR']
index_keys   = ['ndwi', 'sar_ratio', 'ndvi', 'swir_nir']
clip_ranges  = [(-1, 1), (0, 10), (-1, 1), (0, 5)]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle('Spectral Index Distributions by Class\n(Key: Do Flood & WaterBody separate?)',
             fontsize=13, fontweight='bold')

for ax, idx_name, idx_key, (cmin, cmax) in zip(axes, index_labels, index_keys, clip_ranges):
    data_per_class = []
    labels_plot    = []
    colors_plot    = []
    for cls in [0, 1, 2]:
        vals = np.clip(np.array(index_stats[cls][idx_key]), cmin, cmax)
        # subsample for plotting
        if len(vals) > 5000:
            vals = np.random.choice(vals, 5000, replace=False)
        data_per_class.append(vals)
        labels_plot.append(f'{CLASS_NAMES[cls]}\n(n={len(vals):,})')
        colors_plot.append(CLASS_COLORS[cls])

    bp = ax.boxplot(data_per_class, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], colors_plot):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax.set_xticklabels(labels_plot, fontsize=8)
    ax.set_title(idx_name, fontsize=10)
    ax.set_ylim(cmin, cmax)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('spectral_indices.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: spectral_indices.png')

# Print mean values for quick insight
print('\nMEAN INDEX VALUES BY CLASS:')
print(f'{"Index":<20} {"NoFlood":>12} {"Flood":>12} {"WaterBody":>12}')
for idx_name, idx_key in zip(index_labels, index_keys):
    means = [np.mean(index_stats[c][idx_key]) for c in [0,1,2]]
    print(f'{idx_name:<20} {means[0]:>12.4f} {means[1]:>12.4f} {means[2]:>12.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 7: NaN / NoData / Corruption Check
# ═══════════════════════════════════════════════════════════════════

print('═' * 55)
print('  NaN / NODATA / CORRUPTION CHECK')
print('═' * 55)

issues = []
for pid in labelled_ids:
    img, lbl = load_image_label(pid, img_map, lbl_map)
    if img is None:
        issues.append({'patch': pid, 'issue': 'File not found'})
        continue

    nan_count  = int(np.isnan(img).sum())
    inf_count  = int(np.isinf(img).sum())
    zero_bands = [CHANNEL_NAMES[i] for i in range(img.shape[0])
                  if img[i].max() == img[i].min()]
    bad_labels = int(np.isin(lbl, [0, 1, 2], invert=True).sum())

    if any([nan_count > 0, inf_count > 0, zero_bands, bad_labels > 0]):
        issues.append({
            'patch': pid,
            'NaNs': nan_count,
            'Infs': inf_count,
            'ConstantBands': zero_bands,
            'BadLabels': bad_labels
        })

if issues:
    print(f'  ⚠️  Found {len(issues)} patches with potential issues:')
    for issue in issues:
        print(f'  {issue}')
else:
    print('  ✅ All patches clean — no NaN, Inf, or bad label values found!')

# Check test/prediction patches too
print('\n  Checking prediction/test patches...')
pred_issues = []
for f in all_preds:
    with rasterio.open(f) as src:
        arr = src.read().astype(np.float32)
    if np.isnan(arr).any() or np.isinf(arr).any():
        pred_issues.append(os.path.basename(f))
if pred_issues:
    print(f'  ⚠️  Pred patches with issues: {pred_issues}')
else:
    print(f'  ✅ All {len(all_preds)} prediction patches look clean')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 8: CHANNEL CORRELATION MATRIX
# ═══════════════════════════════════════════════════════════════════
# Are channels redundant? Can we see SAR vs optical independence?

channel_samples = {name: [] for name in CHANNEL_NAMES}

for pid in labelled_ids[:20]:  # 20 patches enough for correlation
    img, lbl = load_image_label(pid, img_map, lbl_map)
    if img is None:
        continue
    # Sample 500 random pixels per patch
    h, w = img.shape[1], img.shape[2]
    ridx = np.random.randint(0, h, 500)
    cidx = np.random.randint(0, w, 500)
    for i, name in enumerate(CHANNEL_NAMES):
        channel_samples[name].extend(img[i][ridx, cidx].tolist())

corr_df = pd.DataFrame(channel_samples).corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_df.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(CHANNEL_NAMES)))
ax.set_yticks(range(len(CHANNEL_NAMES)))
ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(CHANNEL_NAMES, fontsize=10)
for i in range(len(CHANNEL_NAMES)):
    for j in range(len(CHANNEL_NAMES)):
        ax.text(j, i, f'{corr_df.values[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(corr_df.values[i, j]) > 0.6 else 'black')
ax.set_title('Channel Correlation Matrix\n(SAR vs Optical independence visible?)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('channel_correlation.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: channel_correlation.png')

print('\nKEY CORRELATIONS (SAR HH vs optical bands):')
for name in CHANNEL_NAMES:
    if 'SAR' not in name:
        print(f'  SAR_HH vs {name:<8}: {corr_df.loc["SAR_HH", name]:.3f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 9: FLOOD vs WATERBODY SEPARABILITY
# The core Phase 2 challenge
# ═══════════════════════════════════════════════════════════════════

# For each channel, compare mean value in Flood vs WaterBody pixels
class_channel_means = {cls: {name: [] for name in CHANNEL_NAMES} for cls in [0, 1, 2]}

for pid in labelled_ids:
    img, lbl = load_image_label(pid, img_map, lbl_map)
    if img is None:
        continue
    for cls in [0, 1, 2]:
        mask = (lbl == cls)
        if mask.sum() < 10:
            continue
        for i, name in enumerate(CHANNEL_NAMES):
            class_channel_means[cls][name].append(np.nanmean(img[i][mask]))

print('═' * 65)
print('  MEAN CHANNEL VALUES BY CLASS  (Flood vs WaterBody separability)')
print('═' * 65)
print(f'{"Channel":<15} {"NoFlood":>10} {"Flood":>10} {"WaterBody":>10}  {"F-W diff":>10}')
print('─' * 65)
for name in CHANNEL_NAMES:
    means = {}
    for cls in [0, 1, 2]:
        vals = class_channel_means[cls][name]
        means[cls] = np.mean(vals) if vals else float('nan')
    diff = means[1] - means[2]  # Flood - WaterBody
    flag = '  ← separable!' if abs(diff) > 100 else ''
    print(f'{name:<15} {means[0]:>10.2f} {means[1]:>10.2f} {means[2]:>10.2f}  {diff:>10.2f}{flag}')

print('\n(Large |F-W diff| → channel helps separate flood from waterbody)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 10: TEST SET SANITY CHECK
# ═══════════════════════════════════════════════════════════════════

print('═' * 55)
print('  TEST / PREDICTION SET SANITY CHECK')
print('═' * 55)
print(f'  Test IDs in test.txt   : {len(test_ids)}')
print(f'  Pred TIFs in pred dir  : {len(all_preds)}')

# Check which test IDs have matching pred TIFs
matched = 0
unmatched = []
for tid in test_ids:
    found = any(tid in k or k in tid for k in pred_map.keys())
    if found:
        matched += 1
    else:
        unmatched.append(tid)

print(f'  Matched (ID → TIF)     : {matched}')
print(f'  Unmatched              : {len(unmatched)}')
if unmatched:
    print(f'  Unmatched IDs: {unmatched}')

# Check one pred image to confirm it has same structure as train images
if all_preds:
    with rasterio.open(all_preds[0]) as src:
        pshape = (src.count, src.height, src.width)
        pdtype = src.dtypes[0]
    print(f'\n  Pred image shape : {pshape} — dtype: {pdtype}')
    print(f'  Train image shape: ({n_bands}, 512, 512)')
    if pshape[0] == n_bands and pshape[1] == 512 and pshape[2] == 512:
        print('  ✅ Pred images match train structure')
    else:
        print('  ⚠️  Pred images differ from train — CHECK THIS!')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 11: SUBMISSION FORMAT VERIFICATION
# ═══════════════════════════════════════════════════════════════════
# Let's write a dummy submission to verify the RLE format is correct

def mask_to_rle(mask):
    """
    Convert binary mask to RLE string.
    Column-major order (Kaggle convention): top-to-bottom, then left-to-right.
    Pixel indexing starts at 1.
    Returns '0 0' if no positive pixels.
    """
    # Flatten column-major (Fortran order)
    flat = mask.flatten(order='F').astype(np.uint8)
    if flat.sum() == 0:
        return '0 0'

    # Find run starts and lengths
    padded = np.concatenate([[0], flat, [0]])
    diffs = np.diff(padded.astype(np.int8))
    starts = np.where(diffs == 1)[0]  # 0-indexed
    ends   = np.where(diffs == -1)[0] # 0-indexed exclusive
    lengths = ends - starts
    starts_1indexed = starts + 1      # Kaggle is 1-indexed

    rle_pairs = []
    for s, l in zip(starts_1indexed, lengths):
        rle_pairs.append(f'{s} {l}')
    return ' '.join(rle_pairs)

# Test on a real label
img, lbl = load_image_label(labelled_ids[0], img_map, lbl_map)
flood_mask = (lbl == 1).astype(np.uint8)
rle_str = mask_to_rle(flood_mask)
print(f'Test RLE encoding:')
print(f'  Flood pixels in mask : {flood_mask.sum():,}')
print(f'  RLE string (first 80 chars): {rle_str[:80]}...')

# Verify decode matches original
def rle_to_mask(rle_str, shape=(512, 512)):
    if rle_str == '0 0' or rle_str.strip() == '':
        return np.zeros(shape, dtype=np.uint8)
    nums = list(map(int, rle_str.split()))
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for i in range(0, len(nums), 2):
        start, length = nums[i] - 1, nums[i+1]
        flat[start:start + length] = 1
    return flat.reshape(shape, order='F')

decoded = rle_to_mask(rle_str)
match = np.array_equal(flood_mask, decoded)
print(f'  Encode → Decode match: {"✅ YES" if match else "❌ NO — BUG!"}')

In [ ]:
# ── Create dummy all-zero submission to verify format ──────────────
rows = []
for tid in test_ids:
    rows.append({'id': tid, 'rle_mask': '0 0'})

dummy_sub = pd.DataFrame(rows)
dummy_sub.to_csv('dummy_submission.csv', index=False)
print('Dummy submission saved: dummy_submission.csv')
print(dummy_sub.head())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FINAL REPORT
# ═══════════════════════════════════════════════════════════════════
print('=' * 65)
print('  COMPLETE EDA SUMMARY — COPY THIS INTO YOUR STRATEGY DOC')
print('=' * 65)

print(f'''
📦 DATA STRUCTURE
  Train patches     : {len(train_ids)}
  Val   patches     : {len(val_ids)}
  Test  patches     : {len(test_ids)}
  Image size        : 512 x 512 pixels
  Bands per image   : {n_bands}
  Band names        : {CHANNEL_NAMES}
  Label values      : 0=NoFlood, 1=Flood, 2=WaterBody

📊 CLASS IMBALANCE
  NoFlood    : {global_counts[0]/total_pixels*100:.1f}%  of all pixels
  Flood      : {global_counts[1]/total_pixels*100:.1f}%  of all pixels  ← RARE
  WaterBody  : {global_counts[2]/total_pixels*100:.1f}%  of all pixels

⚠️  MODELLING IMPLICATIONS
  1. Class weights ESSENTIAL — flood is severely under-represented
  2. Use Dice + CrossEntropy loss to handle imbalance
  3. Monitor per-class IoU, NOT overall accuracy
  4. NDWI separates water (flood+WB) from land well
  5. SAR channels are {'LOW' if corr_df.loc['SAR_HH','NIR'] < 0.4 else 'HIGH'}
     correlated with optical → dual encoder justified
  6. Submission format: RLE column-major, 1-indexed, '0 0' for empty
''')
print('=' * 65)
print('  ✅ EDA COMPLETE. Outputs saved as PNG files + dummy_submission.csv')
print('=' * 65)